# Exercise B2 — Worked PANN Examples

**Companion to Lecture B (Section B.5)** and to Supplement S2 of the lecture notes.
Do this after Exercise B1.

A *physics-augmented* network puts the physics into the **architecture** (by construction),
not into the loss (by persuasion). Here: learn a 1D material law $\sigma(\varepsilon)$ from data such
that — for *any* weights — (i) stress derives from an energy, (ii) the tangent stiffness is never
negative, (iii) zero strain gives zero stress. Fill in every `...`; run the ✅ checks.

---
*Author: Ramses Sala · Copyright © 2025 Ramses Sala · Part of the course* Essential Concepts in FEM and Physics-Based ML. *Created with assistance of Claude Code for text, image and code refinement and generation.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

# ---------- helpers (same as in Exercises A and B1) ----------
# num_grad is a transparent stand-in for automatic differentiation: real frameworks
# compute the same numbers exactly and much faster.
def num_grad(f, th, h=1e-6):
    """Numerical gradient of scalar f at parameter vector th (central differences)."""
    g = np.zeros_like(th)
    for i in range(len(th)):
        tp = th.copy(); tm = th.copy(); tp[i] += h; tm[i] -= h
        g[i] = (f(tp) - f(tm)) / (2*h)
    return g

def adam_minimize(f, th0, steps=2000, lr=0.02, verbose_every=None):
    """Adam: gradient descent with per-parameter step adaptation."""
    th = np.asarray(th0, float).copy()
    m = np.zeros_like(th); v = np.zeros_like(th)
    for k in range(steps):
        g = num_grad(f, th)
        m = 0.9*m + 0.1*g
        v = 0.999*v + 0.001*g*g
        mh = m/(1-0.9**(k+1)); vh = v/(1-0.999**(k+1))
        th -= lr*mh/(np.sqrt(vh)+1e-8)
        if verbose_every and k % verbose_every == 0:
            print(f"step {k:5d}   J = {f(th):.4e}")
    return th


---
## Part 1 (Example P1) — Hooke's law is the smallest PANN

Energy $\psi = \tfrac12\theta^2\varepsilon^2$, hence $\sigma = \theta^2\varepsilon$ with $E=\theta^2 \ge 0$ by construction.

1. **On paper**: minimize $J(E) = \sum_i (E\varepsilon_i - \sigma_i)^2$ and show $E = \sum\sigma_i\varepsilon_i/\sum\varepsilon_i^2$.
2. Evaluate for the three tensile readings below (strain in –, stress in MPa).
3. Why do we train $\theta$ rather than $E$ itself?

In [ ]:
eps_d = np.array([0.001, 0.002, 0.003])
sig_d = np.array([205.0, 395.0, 610.0])      # MPa
E = ...                                       # TODO: closed-form least squares
theta = ...                                   # TODO: the trainable weight behind E
print(f"E = {E/1000:.1f} GPa   theta = {theta:.2f}")

In [ ]:
# ✅ check — Part 1
assert abs(E - 2.825/14e-6) < 1e-3, "Expect E = 2.825 / 14e-6 MPa = 201.8 GPa."
assert abs(theta**2 - E) < 1e-6
print("OK. You trained a physics-augmented material model — and it is Young's modulus.")

---
## Part 2 (Example P2) — The one-layer PANN, and a proof by pencil

Energy $\psi(\varepsilon) = \sum_j \alpha_j^2\ln\cosh(w_j\varepsilon + b_j)$ minus the correction
$\psi(0) + \psi'(0)\varepsilon$. Its stress:
$$\hat\sigma(\varepsilon) = \sum_j \alpha_j^2 w_j\bigl[\tanh(w_j\varepsilon + b_j) - \tanh(b_j)\bigr].$$

1. **On paper** (one line each, for *arbitrary* weights): show $\hat\sigma(0) = 0$ and
   $\hat\sigma'(\varepsilon) = \sum_j\alpha_j^2 w_j^2\,\mathrm{sech}^2(w_j\varepsilon+b_j) \ge 0$.
2. Implement $\hat\sigma$ and its tangent $\hat\sigma'$ (parameter vector `th = [alpha (H), w (H), b (H)]`).
3. Stress-test your proof: for 200 *random* weight vectors, confirm numerically that both
   properties hold. (No training has happened — that is the point.)

In [ ]:
H = 4
def sigma_pann(e, th):
    a = th[:H]; w = th[H:2*H]; b = th[2*H:3*H]
    return ...                      # TODO: stress of the PANN (see formula)

def tangent_pann(e, th):
    a = th[:H]; w = th[H:2*H]; b = th[2*H:3*H]
    return ...                      # TODO: d sigma / d eps

rng = np.random.default_rng(42)
ee = np.linspace(-2, 2, 401)
worst_origin, worst_tangent = 0.0, np.inf
for trial in range(200):
    th_rand = rng.normal(0, 2, 3*H)
    worst_origin  = max(worst_origin, abs(sigma_pann(np.array([0.0]), th_rand)[0]))
    worst_tangent = min(worst_tangent, tangent_pann(ee, th_rand).min())
print(f"largest |sigma(0)| over 200 random nets: {worst_origin:.1e}")
print(f"smallest tangent stiffness over 200 random nets: {worst_tangent:.3e}")

In [ ]:
# ✅ check — Part 2
assert worst_origin < 1e-12, "sigma(0) must vanish for ANY weights."
assert worst_tangent >= 0.0, "The tangent stiffness must be non-negative for ANY weights."
print("OK. Untrained, random, and already physically admissible: that is 'by construction'.")

---
## Part 3 (Example P3) — Constrained versus unconstrained, honestly

Ten noisy tensile readings ($0.05\le\varepsilon\le0.5$; no compression data, nothing at the origin) of a
saturating material $\sigma=\tanh(3\varepsilon)$. Fit two models with the same optimizer:
an **unconstrained** network (given) and your **PANN**.

1. Write both mean-squared losses and train (3000 Adam steps each).
2. Fill the comparison: data misfit, $\sigma(0)$, smallest tangent on $[-0.5, 1]$, error at $\varepsilon=1$.
3. Plot both against the truth on $[-0.5, 1]$. Where exactly does the unconstrained model go wrong?

In [ ]:
# given: data, and the unconstrained competitor
rng = np.random.default_rng(0)
e_d = np.linspace(0.05, 0.5, 10)
s_d = np.tanh(3*e_d) + 0.02*rng.standard_normal(10)

def sigma_nn(e, th):          # th = [c (H), w (H), b (H), c0]  -- no physics built in
    c = th[:H]; w = th[H:2*H]; b = th[2*H:3*H]; c0 = th[3*H]
    return np.tanh(np.outer(e, w) + b) @ c + c0

th0_nn   = np.concatenate([rng.normal(0, .5, H), rng.normal(0, 2, H), rng.normal(0, 1, H), [0.]])
th0_pann = np.concatenate([rng.normal(0, .7, H), rng.normal(0, 2, H), rng.normal(0, 1, H)])

In [ ]:
loss_nn   = lambda th: ...       # TODO: mean squared misfit of sigma_nn
loss_pann = lambda th: ...       # TODO: mean squared misfit of sigma_pann
th_nn   = adam_minimize(loss_nn,   th0_nn,   steps=3000, lr=0.02)
th_pann = adam_minimize(loss_pann, th0_pann, steps=3000, lr=0.02)

ee = np.linspace(-0.5, 1.0, 601); zero = np.array([0.0]); one = np.array([1.0])
nn_origin   = sigma_nn(zero, th_nn)[0]
pann_origin = sigma_pann(zero, th_pann)[0]
nn_min_tan   = ...               # TODO: smallest slope of sigma_nn on ee (np.gradient)
pann_min_tan = ...               # TODO: smallest tangent of the PANN on ee
print(f"misfit      NN {loss_nn(th_nn):.1e}   PANN {loss_pann(th_pann):.1e}")
print(f"sigma(0)    NN {nn_origin:+.4f}   PANN {pann_origin:+.1e}")
print(f"min tangent NN {nn_min_tan:+.3f}   PANN {pann_min_tan:+.3f}")
print(f"err at 1.0  NN {abs(sigma_nn(one, th_nn)[0]-np.tanh(3)):.3f}   PANN {abs(sigma_pann(one, th_pann)[0]-np.tanh(3)):.3f}")
plt.axvspan(0.05, 0.5, color='0.92', label='data range')
plt.plot(ee, np.tanh(3*ee), 'k', lw=1, label='truth')
plt.plot(ee, sigma_nn(ee, th_nn), 'r', label='unconstrained')
plt.plot(ee, sigma_pann(ee, th_pann), 'b--', label='PANN')
plt.plot(e_d, s_d, 'k.'); plt.xlabel('strain'); plt.ylabel('stress'); plt.legend(); plt.show()

In [ ]:
# ✅ check — Part 3
assert abs(pann_origin) < 1e-12 and pann_min_tan >= 0, "The PANN guarantees must survive training."
assert abs(nn_origin) > 1e-4, "The unconstrained net almost surely predicts stress at zero strain."
print("OK. Same data, same optimizer, same fit quality — different physics.")
print("(Reference run: unconstrained sigma(0) = +0.016, min tangent = -0.73 just beyond the data.)")

### Part 3b — What the constraints do *not* buy; and one more constraint

1. **Honesty experiment** (no check): regenerate the data with `default_rng(1)` and `default_rng(3)`
   and refit. The PANN stays admissible — but is its *extrapolation* into compression always
   accurate? What does "by construction" guarantee, and what not?
2. The material is symmetric: $\sigma(-\varepsilon) = -\sigma(\varepsilon)$. Build that in by setting $b_j = 0$:
   $\hat\sigma = \sum_j\alpha_j^2 w_j\tanh(w_j\varepsilon)$. Fit it to the same tension-only data and
   evaluate the error at $\varepsilon = -0.3$ — a strain with **no** data anywhere near it.

In [ ]:
def sigma_odd(e, th):            # th = [alpha (H), w (H)]
    a = th[:H]; w = th[H:2*H]
    return ...                   # TODO: symmetric PANN

rng2 = np.random.default_rng(10)
th_odd = adam_minimize(lambda th: np.mean((sigma_odd(e_d, th) - s_d)**2),
                       np.concatenate([rng2.normal(0, .7, H), rng2.normal(0, 2, H)]), steps=3000, lr=0.02)
m03 = np.array([-0.3])
err_odd  = abs(sigma_odd(m03, th_odd)[0]  - np.tanh(-0.9))
err_pann = abs(sigma_pann(m03, th_pann)[0] - np.tanh(-0.9))
err_nn   = abs(sigma_nn(m03, th_nn)[0]    - np.tanh(-0.9))
print(f"error at eps = -0.3:   unconstrained {err_nn:.3f}   PANN {err_pann:.3f}   symmetric PANN {err_odd:.3f}")

In [ ]:
# ✅ check — Part 3b
assert err_odd < 0.05, "With symmetry built in, compression comes out right without compression data."
print("OK. Every requirement moved into the architecture is one the data need not teach.")

---
## Part 4 (Example P4) — The learned law at work: a spring that asks a network

A bar ($A = L = 1$, fixed at one end) of *your trained PANN material* is pulled by $F = 0.6$.
Equilibrium: $r(u) = A\,\hat\sigma(u/L) - F = 0$ — Lecture 1's $ku=f$ with a $u$-dependent stiffness.

**Newton's method**: repeat $u \leftarrow u - r(u)/k_t(u)$ with the tangent stiffness
$k_t = A\,\hat\sigma'(u/L)/L$. Each iteration is one *linear* spring solve.

1. Implement the loop (start from $u=0$); print $u$, $r$, $k_t$ per iteration.
2. Compare with the true material's answer $u = \operatorname{artanh}(0.6)/3 \approx 0.2310$.
3. Why does the guarantee $k_t \ge 0$ from Part 2 matter *here*?

In [ ]:
F, A, L = 0.6, 1.0, 1.0
u = 0.0
for it in range(8):
    r  = ...        # TODO: residual, using sigma_pann with th_pann
    kt = ...        # TODO: tangent stiffness, using tangent_pann
    print(f"it {it}: u = {u:.5f}   r = {r:+.2e}   k_t = {kt:.4f}")
    if abs(r) < 1e-12: break
    u = ...         # TODO: Newton update
print("true material would give u =", np.arctanh(0.6)/3)

In [ ]:
# ✅ check — Part 4
assert abs(A*sigma_pann(np.array([u/L]), th_pann)[0] - F) < 1e-9, "Newton should have converged."
assert abs(u - np.arctanh(0.6)/3) < 0.02, "The learned law should reproduce the true response inside the data range."
print("OK. One arrow of the diagram replaced by a network; statics and solver untouched.")

---
## Closing reflections

1. Compare with Exercise B1, Part 6 (damage detection): what was *assumed* there that is *learned* here?
   What kind of data did each need?
2. Transpose to heat: you want to learn a temperature-dependent conductivity $\lambda(T)$ from
   experiments. Which requirement plays the role of 'tangent stiffness $\ge 0$'? How would you build
   it into a network?
3. In 1D every continuous $\sigma(\varepsilon)$ has an energy. Why is 'energy first' nonetheless the
   right habit to form now? (What changes when stress and strain have several components?)